# Quill — Notebook 06: Quantum Interference

In this notebook, we will see that **probability amplitudes can reinforce or cancel**. We will use one bit of `Knot("H")` and watch the decoded character.

We will keep the experiment small: the least-significant bit of ASCII `H` is `0`. Applying a Hadamard creates a superposition between `H` and `I`. A phase flip with `Z`, followed by another Hadamard, changes how those amplitudes combine.

## 1. The idea: amplitudes have signs

For a qubit initially in $|0\rangle$, a Hadamard gives

$$
H|0\rangle = \frac{|0\rangle+|1\rangle}{\sqrt{2}}.
$$

The $Z$ gate leaves $|0\rangle$ unchanged and flips the sign of $|1\rangle$:

$$
Z\frac{|0\rangle+|1\rangle}{\sqrt{2}}
=\frac{|0\rangle-|1\rangle}{\sqrt{2}}.
$$

A second Hadamard combines the amplitudes:

$$
H\frac{|0\rangle-|1\rangle}{\sqrt{2}}=|1\rangle.
$$

So the sequence $H \rightarrow Z \rightarrow H$ turns $|0\rangle$ into $|1\rangle$ with certainty in an ideal circuit. The key is the **relative minus sign**: this is interference, not a coin flip.

## 2. Set up Qiskit

In [ ]:
%pip -q install qiskit qiskit-aer

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

simulator = AerSimulator()

## 3. Encode `H` as a Knot

`H` has ASCII value $72$, binary `01001000`. In this notebook, qubit `0` is the least-significant bit (LSB), so it is `0` for `H`. Flipping that bit changes ASCII 72 (`H`) to 73 (`I`).

We prepare the other seven bits of `H` with X gates, then apply interference operations only to qubit `0`.

In [ ]:
def prepare_character(circuit, char, offset=0):
    bits = format(ord(char), "08b")  # displayed most-significant bit first
    for q, bit in enumerate(reversed(bits)):
        if bit == "1":
            circuit.x(offset + q)

# Prepare Knot H in an 8-qubit register.
prep = QuantumCircuit(8, name="Knot H")
prepare_character(prep, "H")
prep.draw("mpl")

## 4. Experiment A — two Hadamards

First, apply $H$ twice to qubit 0. Since $H^2=I$, the qubit returns to $|0\rangle$ and the decoded character should be `H` every time.

In [ ]:
qc_a = QuantumCircuit(8, 8)
prepare_character(qc_a, "H")
qc_a.h(0)
qc_a.h(0)
qc_a.measure(range(8), range(8))

counts_a = simulator.run(transpile(qc_a, simulator), shots=1000).result().get_counts()
print(counts_a)
plot_histogram(counts_a)

## 5. Experiment B — Hadamard, phase flip, Hadamard

Now apply $H \rightarrow Z \rightarrow H$ to qubit 0. The relative phase changes, and the amplitudes combine to give the opposite basis state. For the character register, that means `H` becomes `I` on measurement.

In [ ]:
qc_b = QuantumCircuit(8, 8)
prepare_character(qc_b, "H")
qc_b.h(0)
qc_b.z(0)
qc_b.h(0)
qc_b.measure(range(8), range(8))

counts_b = simulator.run(transpile(qc_b, simulator), shots=1000).result().get_counts()
print(counts_b)
plot_histogram(counts_b)

## 6. Decode measured bitstrings

In [ ]:
def decode_counts(counts):
    decoded = {}
    for bitstring, count in counts.items():
        # Qiskit displays classical bit c7 ... c0; reverse to get the byte value.
        value = int(bitstring.replace(" ", ""), 2)
        char = chr(value)
        decoded[char] = decoded.get(char, 0) + count
    return decoded

print("Experiment A:", decode_counts(counts_a))
print("Experiment B:", decode_counts(counts_b))

## 7. Why measurement timing matters

If you measure immediately after the first Hadamard, you get `H` or `I` with roughly 50/50 probability. But if you **do not measure**, the amplitudes remain coherent and later gates can interfere. Measurement is not just “looking”; it changes what happens next by collapsing the state.

Try this by measuring after the first `H` and then applying more gates in a separate circuit experiment. Compare it with the coherent sequence above.

## 8. The takeaway

- $H$ creates a superposition from a basis state.
- $Z$ changes the phase of the $|1\rangle$ component without changing its measurement probability immediately.
- A later $H$ turns that phase difference into a measurable outcome.
- Interference is about **amplitudes combining**, not characters understanding their own meaning.

**Exercise:** Remove `z(0)` from Experiment B. Predict the outcome, run it, and explain why it returns to `H`.